# RL4CRN tutorial notebook: Habituation hallmarks

This notebook extends the habituation task to jointly evaluate hallmarks 1, 2, 3, 4, 5, 6, and 10. The task keeps the original peak-ratio/log-loss structure and adds an explicit sign `s`: `s=+1` searches for habituation and `s=-1` searches for sensitization.

In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())

## 1) Imports

In [ ]:
from RL4CRN.utils.input_interface import Configurator, make_task, print_task_summary
from RL4CRN.utils.default_tasks.HabituationTaskKind import HabituationHallmarksTaskKind

HabituationHallmarksTaskKind.pretty_help()

## 2) Template IO/CRN

In [ ]:
from RL4CRN.utils.crn_builders import build_simple_IOCRN

cfg = Configurator.preset("paper")
cfg.solver.algorithm = "CVODE"
cfg.solver.rtol = 1e-6
cfg.solver.atol = 1e-6

species_labels = ["X_1", "X_2", "X_3", "X_4"]
crn, species_labels = build_simple_IOCRN(
    species=species_labels,
    production_input_map={"X_1": "u_1"},
    degradation_input_map={},
    dilution_map={"X_1": 0.1, "X_2": 0.1, "X_3": 0.1, "X_4": 0.1,},
    production_map={"X_2": 0.1, "X_3": 0.1},
    output_species="X_4",
    solver=cfg.solver,
)

print("Template CRN built.")
print("num_inputs:", crn.num_inputs)
print("species:", species_labels)

## 3) Reaction library

In [ ]:
from RL4CRN.utils.library_builders import build_MAK_library

library_components = build_MAK_library(crn, species_labels, order=2)
library, M, K, masks = library_components
print("Library built: M=", M, "K=", K)

## 4) Hallmark task

The per-train score uses the paper-style ratio term

$$H_s(P)=\log\left(\max_i \left(\frac{p_{i+1}}{p_i+\epsilon}\right)^s + \epsilon\right),$$

with `s=+1` for habituation and `s=-1` for sensitization. Additional penalties are added for recovery, potentiation across trains, frequency sensitivity, intensity sensitivity, subliminal accumulation, and long-term memory.

In [ ]:
t_on = 1.0
periods = [5.0, 10.0, 15.0]
pulse_shapes = [(t_on, P - t_on) for P in periods]

task = make_task(
    template_crn=crn,
    library_components=library_components,
    kind="habituation_hallmarks",
    species_labels=species_labels,
    params={
        "pulse_shapes": pulse_shapes,
        "n_repeats_pre": 10,
        "n_repeats_post": 10,
        "gap_time": 50.0,
        "n_t": 1000,
        "ic": "from_ss",
        "max_peak": 10.0,
        "min_peak": 0.1,
        "u_values": [1.0],
        "s": 1,
        "habituation_weight": 2.0,
        "early_peak_sep_weight": 10.0,
        "early_peak_min_change": 0.08,
        "early_peak_count": 5,
        "freq_weight": 1.0,
        "gap_weight": 5.0,
        "recovery_tol": 0.05,
        "potentiation_blocks": 3,
        "potentiation_gap_time": 50.0,
        "potentiation_weight": 1.0,
        "intensity_values": [1.0, 0.66, 0.33],
        "intensity_weight": 1.0,
        "long_train_repeats": 20,
        "short_gap_time": 10.0,
        "subliminal_weight": 1.0,
        "long_gap_times": [50.0, 100.0, 200.0],
        "long_term_weight": 1.0,
        "long_term_min_memory": 0.05,
        "memory_difference_weight": 1.0,
        "state_reset_weight": 1.0,
        "state_reset_tol": 0.2,
        "state_reset_floor": 1.0,
        "peak_tol": 0.1,
    },
)

print_task_summary(task)
assert len(task.u_list[0]) == crn.num_inputs

## 5) Quick smoke evaluation of the template

In [ ]:
loss, info = task.compute_reward(crn)
print("template loss:", loss)
print("components:")
for k, v in crn.last_task_info["component_losses"].items():
    print(f"  {k}: {v:.4g}")
print("stored hallmark runs:", len(crn.last_task_info["hallmark_runs"]))

In [ ]:
fig, axes = crn.plot_habituation_hallmarks(
    normalize=True,
    plot_cfg={"figsize": (6, 18), "alpha": 0.9},
)

In [ ]:
fig_diag, axes_diag = crn.plot_habituation_hallmark_diagnostics(
    plot_cfg={"figsize": (8, 6)},
)

In [ ]:
fig_int, axes_int = crn.plot_habituation_hallmarks(
    normalize=True,
    group_filter="5 intensity",
    plot_cfg={"figsize": (6, 5), "alpha": 0.9},
)

## 6) Training configuration

In [ ]:
cfg.train.max_added_reactions = 5
cfg.train.epochs = 301
cfg.train.render_every = 5
cfg.train.seed = 0

cfg.render.n_best = 10
cfg.render.disregarded_percentage = 0.9
cfg.render.mode = {
    "style": "logger",
    "task": "habituation_hallmarks",
    "format": "image",
    "topology": True,
    "hof_n_best": 10,
    "normalize": True,
    "plot_cfg": {"figsize": (6, 18), "alpha": 0.9},
    "diagnostics": True,
    "diagnostics_plot_cfg": {"figsize": (8, 6)},
    "group_plots": ["4 frequency", "5 intensity", "10 long-term"],
    "group_plot_cfg": {"figsize": (6, 5), "alpha": 0.9},
}

## 7) Logger and trainer

In [ ]:
from datetime import datetime
from RL4CRN.utils.input_interface import make_session_and_trainer

logger = None
if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from pytorch_lightning.loggers import CometLogger
    task_name = "Habituation_hallmarks_Task"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    logger = CometLogger(
        api_key=os.environ["COMET_API_KEY"],
        project=task_name,
        workspace=os.environ["COMET_WORKSPACE"],
        name=f"{task_name}_{timestamp}",
    ).experiment

trainer = make_session_and_trainer(cfg, task, logger=logger)

## 8) Train and inspect

In [ ]:
checkpoint_path = "habituation_hallmarks_task_chkpt.pkl"
trainer.run(epochs=cfg.train.epochs, checkpoint_path=checkpoint_path)

In [ ]:
best = trainer.inspect_best(plot=True, normalize=True)
if best is not None:
    print("Best loss:", best.last_task_info.get("reward"))
    print("Component losses:", best.last_task_info.get("component_losses"))

In [ ]:
from RL4CRN.utils.visualizations import topology_graph

hof_envs = list(trainer.s.mult_env.hall_of_fame or [])[:10]
print("HoF entries plotted:", len(hof_envs))

for i, env in enumerate(hof_envs):
    crn_hof = env.state
    print(f"HoF {i} loss:", crn_hof.last_task_info.get("reward"))
    fig, _ = crn_hof.plot_habituation_hallmarks(
        normalize=True,
        plot_cfg={"figsize": (6, 18), "alpha": 0.9},
    )
    fig.suptitle(f"HoF {i} Habituation Hallmarks")
    fig_diag, _ = crn_hof.plot_habituation_hallmark_diagnostics(
        plot_cfg={"figsize": (8, 6)},
    )
    fig_diag.suptitle(f"HoF {i} Loss Diagnostics")

if hof_envs:
    fig_hof_div = topology_graph([env.state for env in hof_envs], t=5, figsize=(10, 10))
    fig_hof_div.suptitle("HoF Top-10 Diversity Graph")

current_top_envs = sorted(
    trainer.s.mult_env.envs,
    key=lambda env: env.state.last_task_info.get("reward", float("inf")),
)[:10]
if current_top_envs:
    fig_batch_div = topology_graph([env.state for env in current_top_envs], t=5, figsize=(10, 10))
    fig_batch_div.suptitle("Current Batch Top-10 Diversity Graph")

In [ ]:
trainer.save(checkpoint_path)